# Advanced PyTorch — Functorch, Custom Ops, and More

**What you'll learn:**
- `torch.func` transforms: vmap, grad, jacobian, hessian
- Per-sample gradients for privacy-preserving ML
- Custom operators with `torch.library`
- Sparse tensors, complex numbers, and FFT
- Profiling and meta-device analysis

**Prerequisites:** Comfortable with PyTorch tensors, autograd, and nn.Module

In [ ]:
import torch
import torch.nn as nn
from torch.func import vmap, grad, jacrev, jacfwd, hessian
import math
print(f"PyTorch version: {torch.__version__}")
print(f"torch.func transforms: vmap, grad, jacrev, jacfwd, hessian")

## Part 1: torch.func.vmap — Automatic Batching

`vmap` (vectorized map) transforms a function that operates on single samples into one that operates on batches. Instead of writing explicit batch loops or reshaping, you write the per-sample logic and let `vmap` handle batching.

**Why this matters:** Many operations are naturally expressed per-sample (per-sample loss, per-sample gradients), but you need batch efficiency for training.

In [ ]:
# Per-sample cosine similarity — the natural way to think about it
def cosine_sim_single(x, y):
    """Cosine similarity for a single pair of vectors."""
    return torch.dot(x, y) / (x.norm() * y.norm())

# Without vmap: manual loop over batch
x_batch = torch.randn(8, 128)
y_batch = torch.randn(8, 128)

# Slow: Python loop
results_loop = torch.stack([cosine_sim_single(x_batch[i], y_batch[i]) for i in range(8)])
print(f"Loop result shape: {results_loop.shape}")
print(f"Loop results: {results_loop[:4]}")

# Fast: vmap handles the batching
batched_cosine = vmap(cosine_sim_single)
results_vmap = batched_cosine(x_batch, y_batch)
print(f"\nvmap result shape: {results_vmap.shape}")
print(f"vmap results: {results_vmap[:4]}")
print(f"\nResults match: {torch.allclose(results_loop, results_vmap)}")

### vmap with a Per-Sample Loss

A more practical example: computing a per-sample loss for a mini-batch. This is useful for importance sampling, curriculum learning, or debugging which samples are hard.

In [ ]:
# Per-sample MSE loss using vmap
model = nn.Linear(10, 1)
params = dict(model.named_parameters())

def compute_loss_single(x, y, weight, bias):
    """Loss for a SINGLE sample (no batch dimension)."""
    pred = x @ weight.T + bias
    return ((pred - y) ** 2).sum()

# Sample data
X = torch.randn(16, 10)
Y = torch.randn(16, 1)

# vmap over the batch dimension (dim 0 of X and Y), not over params
per_sample_loss = vmap(compute_loss_single, in_dims=(0, 0, None, None))
losses = per_sample_loss(X, Y, params['weight'], params['bias'])

print(f"Per-sample losses shape: {losses.shape}")
print(f"Per-sample losses: {losses[:5]}")
print(f"Mean loss: {losses.mean():.4f}")

# Verify against standard batched loss
batched_loss = nn.functional.mse_loss(model(X), Y, reduction='none').sum(dim=1)
print(f"\nStandard batched loss: {batched_loss[:5]}")
print(f"Results match: {torch.allclose(losses, batched_loss)}")

## Part 2: torch.func.grad — Functional Gradients

`grad` computes gradients of a **function** (not tensors). This is the functional programming approach to differentiation — no `.backward()`, no `.grad` attributes.

In [ ]:
# grad of a mathematical function — no tensors with requires_grad needed
def f(x):
    return torch.sin(x) * torch.exp(-x**2)

# Compute first derivative
df = grad(f)
# Compute second derivative
d2f = grad(grad(f))

x = torch.tensor(1.0)
print(f"f(1.0)   = {f(x):.6f}")
print(f"f'(1.0)  = {df(x):.6f}")
print(f"f''(1.0) = {d2f(x):.6f}")

# Evaluate derivatives over a range
import torch
xs = torch.linspace(-3, 3, 7)
print(f"\nx values: {xs.tolist()}")
print(f"f(x):     {[f'{f(xi):.3f}' for xi in xs]}")
print(f"f'(x):    {[f'{df(xi):.3f}' for xi in xs]}")
print(f"f''(x):   {[f'{d2f(xi):.3f}' for xi in xs]}")

## Part 3: Jacobians and Hessians

For many applications (physics-informed ML, second-order optimization, sensitivity analysis), you need full Jacobian or Hessian matrices. `torch.func` provides efficient implementations.

- `jacrev`: Jacobian via reverse-mode AD (efficient when output << input)
- `jacfwd`: Jacobian via forward-mode AD (efficient when input << output)
- `hessian`: Full Hessian matrix (uses jacrev of jacrev)

In [ ]:
# Jacobian: derivative of vector-valued function
def multi_output(x):
    """R^3 -> R^3 function."""
    return torch.stack([
        x[0]**2 + x[1],
        x[0] * x[1] * x[2],
        torch.sin(x[2]),
    ])

x = torch.tensor([1.0, 2.0, 3.0])

# Jacobian matrix (3x3)
J_rev = jacrev(multi_output)(x)
J_fwd = jacfwd(multi_output)(x)

print("Jacobian (reverse mode):")
print(J_rev)
print(f"\nJacobians match: {torch.allclose(J_rev, J_fwd)}")

# Hessian of a scalar function
def scalar_fn(x):
    return (x**3).sum()

H = hessian(scalar_fn)(x)
print(f"\nHessian of sum(x^3) at [1,2,3]:")
print(H)
print(f"Diagonal: {H.diag()} (should be 6*x = [6, 12, 18])")

## Part 4: Per-Sample Gradients (vmap + grad)

The killer combo: `vmap(grad(...))` gives you **per-sample gradients** — the gradient of the loss with respect to model parameters for each sample independently. This is essential for:
- **Differentially Private SGD (DP-SGD)**: clip per-sample gradients before aggregation
- **Influence functions**: measure how each training sample affects the model

In [ ]:
# Per-sample gradients: the functional approach
from torch.func import functional_call

model = nn.Linear(5, 3)
params = {k: v.detach() for k, v in model.named_parameters()}

def compute_loss(params, x, y):
    """Loss for a single (x, y) pair."""
    pred = functional_call(model, params, (x.unsqueeze(0),))
    return nn.functional.cross_entropy(pred, y.unsqueeze(0))

# Gradient of loss w.r.t. params for a single sample
grad_fn = grad(compute_loss)

# vmap over the batch: per-sample gradients!
X = torch.randn(8, 5)
Y = torch.randint(0, 3, (8,))

per_sample_grads = vmap(grad_fn, in_dims=(None, 0, 0))(params, X, Y)

print("Per-sample gradient shapes:")
for name, g in per_sample_grads.items():
    print(f"  {name}: {g.shape}")
    # First dim is batch (8), rest is parameter shape

print(f"\nPer-sample grad norms (weight):")
norms = per_sample_grads['weight'].flatten(1).norm(dim=1)
print(f"  {norms}")
print(f"\nThis is what DP-SGD clips before averaging!")

## Part 5: Custom Operators with torch.library

When you need operations that PyTorch doesn't provide, you can define custom operators that integrate with autograd, torch.compile, and torch.export.

In [ ]:
# Define a custom operator using torch.library
from torch.library import Library, impl

# Create a custom library
custom_lib = Library("myops", "DEF")

# Define the operator signature
custom_lib.define("gelu_approx(Tensor x) -> Tensor")

# Implement for CPU
@impl(custom_lib, "gelu_approx", "CPU")
def gelu_approx_cpu(x):
    """Fast GELU approximation (tanh version)."""
    return 0.5 * x * (1.0 + torch.tanh(
        math.sqrt(2.0 / math.pi) * (x + 0.044715 * x.pow(3))
    ))

# Implement for Meta (shape inference only, no computation)
@impl(custom_lib, "gelu_approx", "Meta")
def gelu_approx_meta(x):
    return torch.empty_like(x)

# Use the custom op
x = torch.randn(4, 8)
result = torch.ops.myops.gelu_approx(x)
print(f"Custom GELU input shape: {x.shape}")
print(f"Custom GELU output shape: {result.shape}")
print(f"\nComparison with PyTorch GELU:")
official = nn.functional.gelu(x, approximate='tanh')
print(f"Max difference: {(result - official).abs().max():.2e}")

## Part 6: Sparse Tensors

Sparse tensors store only non-zero elements, saving memory for matrices that are mostly zeros. PyTorch supports COO (Coordinate) and CSR (Compressed Sparse Row) formats.

In [ ]:
# COO format: store (row, col, value) triples
# Create a 1000x1000 matrix with only 50 non-zero values
indices = torch.stack([
    torch.randint(0, 1000, (50,)),  # row indices
    torch.randint(0, 1000, (50,)),  # col indices
])
values = torch.randn(50)
sparse_coo = torch.sparse_coo_tensor(indices, values, (1000, 1000))

print("COO Sparse Tensor:")
print(f"  Shape: {sparse_coo.shape}")
print(f"  Non-zero elements: {sparse_coo._nnz()}")
print(f"  Density: {sparse_coo._nnz() / (1000*1000) * 100:.4f}%")

# CSR format: efficient for row-slicing and matrix-vector products
dense_small = torch.randn(5, 5)
dense_small[dense_small.abs() < 1.0] = 0  # Zero out small values
sparse_csr = dense_small.to_sparse_csr()

print(f"\nCSR Sparse Tensor:")
print(f"  Shape: {sparse_csr.shape}")
print(f"  Non-zero: {sparse_csr.values().shape[0]}")
print(f"  Dense original:\n{dense_small}")

# Memory comparison
dense_full = sparse_coo.to_dense()
dense_mem = dense_full.numel() * dense_full.element_size()
sparse_mem = (indices.numel() + values.numel()) * 4  # approximate
print(f"\nMemory comparison (1000x1000 matrix, 50 non-zeros):")
print(f"  Dense:  {dense_mem / 1024:.0f} KB")
print(f"  Sparse: {sparse_mem / 1024:.1f} KB")
print(f"  Ratio:  {dense_mem / sparse_mem:.0f}x")

## Part 7: Complex Numbers and FFT

PyTorch has native support for complex tensors and provides a complete FFT module. This is essential for signal processing, physics simulations, and spectral methods.

In [ ]:
# Complex tensor basics
z = torch.complex(torch.tensor([1.0, 2.0, 3.0]), torch.tensor([4.0, 5.0, 6.0]))
print(f"Complex tensor: {z}")
print(f"Real part: {z.real}")
print(f"Imaginary part: {z.imag}")
print(f"Magnitude: {z.abs()}")
print(f"Phase (angle): {z.angle()}")
print(f"Conjugate: {torch.conj(z)}")

# FFT: create a signal with known frequencies
t = torch.linspace(0, 1, 256)  # 1 second, 256 samples
freq1, freq2 = 5.0, 20.0  # 5 Hz and 20 Hz components
signal = torch.sin(2 * torch.pi * freq1 * t) + 0.5 * torch.sin(2 * torch.pi * freq2 * t)

# Compute FFT
spectrum = torch.fft.rfft(signal)
frequencies = torch.fft.rfftfreq(len(t), d=1.0/256)
magnitudes = spectrum.abs()

# Find peaks
top_k = magnitudes.topk(5)
print(f"\nSignal: sin(2π·5t) + 0.5·sin(2π·20t)")
print(f"Top frequency components:")
for idx in top_k.indices:
    if magnitudes[idx] > 1.0:
        print(f"  {frequencies[idx]:.1f} Hz, magnitude: {magnitudes[idx]:.1f}")

## Part 8: torch.profiler

The profiler helps identify bottlenecks by measuring time spent in each operator. This is essential for optimization.

In [ ]:
# Profile a model's forward pass
model = nn.Sequential(
    nn.Linear(256, 512),
    nn.ReLU(),
    nn.Linear(512, 512),
    nn.ReLU(),
    nn.Linear(512, 10),
)

x = torch.randn(64, 256)

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU],
    record_shapes=True,
) as prof:
    for _ in range(10):
        model(x)

# Print table sorted by CPU time
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=15))

## Part 9: Meta Device for Model Analysis

The `meta` device creates tensors with shape and dtype but **no memory allocation**. This is invaluable for analyzing model architecture without OOM-ing on large models.

In [ ]:
# Analyze a large model on meta device — zero memory!
with torch.device("meta"):
    # This would be ~800MB on CPU/GPU, but uses 0 bytes on meta
    large_model = nn.Sequential(
        nn.Linear(4096, 4096),
        nn.ReLU(),
        nn.Linear(4096, 4096),
        nn.ReLU(),
        nn.Linear(4096, 1000),
    )

print("Large model on meta device (0 memory used):")
total_params = 0
for name, p in large_model.named_parameters():
    size_mb = p.numel() * 4 / 1024 / 1024  # float32
    total_params += p.numel()
    print(f"  {name}: {p.shape} ({size_mb:.1f} MB if materialized)")

print(f"\nTotal: {total_params:,} parameters ({total_params * 4 / 1024 / 1024:.0f} MB)")

# Shape propagation works on meta tensors
meta_input = torch.randn(8, 4096, device="meta")
meta_output = large_model(meta_input)
print(f"\nShape check: {meta_input.shape} → {meta_output.shape} (no actual computation)")

## 🏋️ Try It Yourself: Per-Sample Gradients

**Exercise:** Use `vmap` and `grad` to compute per-sample gradients for a linear model on a regression task. Then clip the per-sample gradients to a max norm of 1.0 (simulating DP-SGD).

Steps:
1. Create a simple `nn.Linear(10, 1)` model
2. Generate random data: X (32, 10), Y (32, 1)
3. Use `vmap(grad(...))` to get per-sample gradients
4. Compute the L2 norm of each sample's gradient
5. Clip gradients exceeding norm 1.0
6. Average the clipped gradients

In [ ]:
# Exercise starter code
model_ex = nn.Linear(10, 1)
params_ex = {k: v.detach() for k, v in model_ex.named_parameters()}

X_ex = torch.randn(32, 10)
Y_ex = torch.randn(32, 1)

def loss_fn(params, x, y):
    pred = x @ params['weight'].T + params['bias']
    return ((pred - y) ** 2).mean()

# Your code here:
# 1. Use grad to get the gradient function
# 2. Use vmap to get per-sample gradients
# 3. Compute per-sample gradient norms
# 4. Clip gradients exceeding norm 1.0
# 5. Average the clipped gradients

# Hint:
per_sample_grad_fn = vmap(grad(lambda p, x, y: loss_fn(p, x.unsqueeze(0), y.unsqueeze(0))), in_dims=(None, 0, 0))
per_sample_grads = per_sample_grad_fn(params_ex, X_ex, Y_ex)

# Compute norms and clip
weight_grads = per_sample_grads['weight']  # (32, 1, 10)
norms = weight_grads.flatten(1).norm(dim=1)
print(f"Per-sample gradient norms (first 8): {norms[:8]}")
clip_factor = torch.clamp(1.0 / norms, max=1.0)
clipped = weight_grads * clip_factor[:, None, None]
avg_clipped = clipped.mean(dim=0)
print(f"Averaged clipped gradient shape: {avg_clipped.shape}")
print(f"Max norm after clipping: {clipped.flatten(1).norm(dim=1).max():.4f}")

## Key Takeaways

1. **vmap** eliminates batch loops — write per-sample logic, get batch efficiency for free

2. **grad** provides functional differentiation — no `.backward()`, no `.grad` attributes, just function in → gradient out

3. **jacrev/jacfwd/hessian** give you full derivative matrices efficiently; choose `jacrev` when outputs are small, `jacfwd` when inputs are small

4. **vmap + grad** is the killer combo for per-sample gradients, essential for DP-SGD and influence functions

5. **torch.library** lets you define custom ops that work with autograd, torch.compile, and export

6. **Sparse tensors** (COO, CSR) save massive memory for matrices with <1% density

7. **Complex tensors + FFT** are first-class citizens — use `torch.fft` for spectral analysis

8. **torch.profiler** reveals which ops dominate runtime — always profile before optimizing

9. **Meta device** lets you analyze model architecture (shapes, parameter counts) without allocating memory